# Executed notebook: E06_genie

Exported from Databricks job run `994447175034582` (task `E06_genie`, task run `886242903537339`).

Result: **SUCCESS** · start 2026-10-07T23:01:01.682000+00:00 · end 2026-10-07T23:05:25.536000+00:00

Run URL: https://fevm-serverless-stable-am1uc2.cloud.databricks.com/?o=7474651880045550#job/755461157363253/run/994447175034582


# E06 · Genie agent on metric views and work orders

**What was built.** Genie agent "Plant Maintenance Agent", defined as code in `src/genie/space_config.py`:
3 metric views + `station_health_current` + `work_orders_current`, text instructions (plant naming, risk bands,
compressed time, where to find what), 4 example SQL queries, 4 sample questions and 10 benchmark questions with
reference SQL. Deployed with `tools/genie_deploy.py`.

**What this notebook proves.** The deployed configuration, and live answers: each benchmark question is asked through
the Genie Conversation API; Genie's generated SQL and the reference SQL are executed back to back and compared.

In [ ]:
%pip install -q "databricks-sdk>=0.81" pg8000
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
from _helpers import *  # noqa: F401,F403
import sys, os, time, json
sys.path.append(os.path.abspath("../.."))
from genie import space_config as cfg

## 1 · Deployed configuration

In [ ]:
sp = w.api_client.do("GET", f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}", query={"include_serialized_space": "true"})
ser = json.loads(sp["serialized_space"])
print("title:", sp["title"], "| warehouse:", sp["warehouse_id"])
print("data sources:", [t["identifier"].split(".", 1)[1] for k in ("tables", "metric_views") for t in ser["data_sources"].get(k, [])])
print("sample questions:", [q["question"][0] for q in ser["config"]["sample_questions"]])
print("example SQL questions:", [q["question"][0] for q in ser["instructions"]["example_question_sqls"]])
print("benchmarks:", len(ser["benchmarks"]["questions"]))
print("\ninstructions:\n" + ser["instructions"]["text_instructions"][0]["content"][0])

title: Plant Maintenance Agent | warehouse: fb9bc265e9f4578a
data sources: ['pdm_core.station_health_current', 'pdm_ops.maintenance_metrics', 'pdm_ops.station_risk_metrics', 'pdm_ops.work_order_metrics', 'pdm_ops.work_orders_current']
sample questions: ['How many open work orders do we have by priority?', 'Why is the riskiest station at risk?', 'Which stations need maintenance soon?', 'Which station types fail most often and what is their MTTR?']
example SQL questions: ['How did average risk develop per plant over the last 15 minutes?', 'How many open work orders are there per plant?', 'Which stations are at high risk right now?', 'What is the MTTR and failure count by station type?']
benchmarks: 10

instructions:
You are a maintenance planning assistant for three plants (Plant North = PLT-N, Plant South = PLT-S, Plant East = PLT-E),


## 2 · Live benchmark: Genie answer vs reference SQL

In [ ]:
def norm(rows):
    out = []
    for r in rows:
        vals = set()
        for v in r:
            try:
                vals.add(round(float(v), 2))
            except (TypeError, ValueError):
                vals.add(None if v is None else str(v))
        out.append(vals)
    return out


results, raw_shown = [], False
for q, ref_sql in cfg.benchmarks(CATALOG):
    t0 = time.time()
    msg = w.genie.start_conversation_and_wait(GENIE_SPACE_ID, q)
    secs = time.time() - t0
    if not raw_shown:  # one complete raw API response, as returned by the Genie Conversation API
        print("RAW Genie message response:\n" + json.dumps(msg.as_dict(), indent=1, default=str)[:4000])
        raw_shown = True
    sql = next((a.query.query for a in (msg.attachments or []) if a.query), None)
    text = next((a.text.content for a in (msg.attachments or []) if a.text and a.text.content), None)
    g_rows = [list(r) for r in spark.sql(sql).collect()] if sql else []
    r_rows = [list(r) for r in spark.sql(ref_sql).collect()]
    ok = bool(sql) and len(g_rows) == len(r_rows) and all(any(rr <= gg for gg in norm(g_rows)) for rr in norm(r_rows))
    results.append((("PASS" if ok else "FAIL"), round(secs, 1), q))
    print(f"\n### {'PASS' if ok else 'FAIL'} ({secs:.1f} s): {q}")
    print("Genie SQL:\n" + (sql or "<none>"))
    print("Genie rows:", g_rows[:5])
    print("Reference rows:", r_rows[:5])
    if text:
        print("Genie text:", text[:400])

RAW Genie message response:
{
 "attachments": [
  {
   "attachment_id": "01f1c2a300b41a879d43c9d4c57beb7d",
   "query": {
    "description": "You want to see a list of all stations that are currently at high risk of failure.",
    "query": "SELECT COUNT(*) AS `high_risk_stations`\nFROM `serverless_stable_am1uc2_catalog`.`pdm_core`.`station_health_current`\nWHERE `risk_band` = 'HIGH'",
    "query_result_metadata": {
     "row_count": 1
    },
    "statement_id": "01f1c2a3-00be-1b07-9165-720eab466ddb",
    "thoughts": [
     {
      "content": "You want to see a list of all stations that are currently at high risk of failure.",
      "thought_type": "THOUGHT_TYPE_DESCRIPTION"
     },
     {
      "content": "- serverless_stable_am1uc2_catalog.pdm_core.station_health_current",
      "thought_type": "THOUGHT_TYPE_DATA_SOURCING"
     },
     {
      "content": "- Filter stations to include only those with a risk band labeled 'HIGH'.\n- Retrieve the station ID, plant ID, station type, and fa

In [ ]:
display(spark.createDataFrame(results, "verdict string, seconds double, question string"))
print(f"{sum(r[0] == 'PASS' for r in results)}/{len(results)} benchmark questions match the reference result")

| verdict | seconds | question |
|---|---|---|
| PASS | 13.6 | How many stations are at high risk right now? |
| PASS | 13.2 | How many stations are currently down? |
| PASS | 18.9 | What are the 5 stations with the highest failure probability right now? |
| PASS | 12.1 | Which station type had the most failures? |
| PASS | 13.3 | What is the mean time to repair in minutes for each plant? |
| PASS | 17.6 | Which failure mode caused the most downtime? |
| PASS | 17.3 | What is the total parts cost per plant? |
| PASS | 18.8 | What share of repairs were preventive for each station type? |
| PASS | 11.8 | How many failures did Plant North have? |
| PASS | 17.5 | List the stations on line PLT-S-B with their current risk band. |


10/10 benchmark questions match the reference result


## 3 · Sample questions (free-form answers)

In [ ]:
for q in cfg.SAMPLE_QUESTIONS:
    msg = w.genie.start_conversation_and_wait(GENIE_SPACE_ID, q)
    sql = next((a.query.query for a in (msg.attachments or []) if a.query), None)
    text = next((a.text.content for a in (msg.attachments or []) if a.text and a.text.content), None)
    print(f"\n### {q}\nGenie text: {text}\nGenie SQL:\n{sql}")
    if sql:
        print("rows:", [list(r) for r in spark.sql(sql).limit(5).collect()])

### Which stations need maintenance soon?
Genie text: **7 stations** currently need maintenance soon: **4 are HIGH risk** and **3 are ELEVATED**. Some of the highest-risk stations are:
- **PLT-S-C08** — **99.6%** risk (**HIGH**), driven by **vibration_rms**
- **PLT-S-D04** — **98.6%** risk (**HIGH**), driven by **bearing_temp_c**
- **PLT-N-C05** — **88.5%** risk (**HIGH**), driven by **vibration_rms**
- **PLT-S-C02** — **87.6%** risk (**HIGH**), driven by **motor_current_a**
- **PLT-N-C01** — **51.5%** risk (**ELEVATED**), driven by **vibration_rms**

Across these 7 stations, risk ranges from **47.6% to 99.6%**, and **Plant South** accounts for **4 of the 7** stations flagged for near-term maintenance.
Genie SQL:
SELECT station_health_current.station_id, station_health_current.plant_id, station_health_current.line_id, station_health_current.station_type, station_health_current.risk_band, ROUND(station_health_current.failure_probability * 100, 1) AS risk_pct, station_health_current.top_